# **Analyse du comportement des clients**

Quelle(s) stratégie(s) marketing adopter pour fidéliser les clients dans un objectif de hausse du chiffre d'affaires ?

# Importation des fichiers et nettoyage des données

## Accès depuis Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


## Importation des packages

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
pd.options.display.float_format = '{:,.2f}'.format
import warnings
warnings.filterwarnings("ignore")

## Lecture des fichiers

In [ ]:
chemin = '/content/gdrive/MyDrive/ProjetRetail/'

retail = pd.read_csv(chemin + 'retail_data.csv')

In [ ]:
pd.set_option('display.max_columns', None)

retail.head()

,Transaction_ID,Customer_ID,Name,Email,Phone,Address,City,State,Zipcode,Country,Age,Gender,Income,Customer_Segment,Date,Year,Month,Time,Total_Purchases,Amount,Total_Amount,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings,products
0,"8,691,788.00","37,249.00",Michelle Harrington,Ebony39@gmail.com,"1,414,786,801.00",3959 Amanda Burgs,Dortmund,Berlin,"77,985.00",Germany,21.00,Male,Low,Regular,9/18/2023,"2,023.00",September,22:03:55,3.00,108.03,324.09,Clothing,Nike,Shorts,Excellent,Same-Day,Debit Card,Shipped,5.00,Cycling shorts
1,"2,174,773.00","69,749.00",Kelsey Hill,Mark36@gmail.com,"6,852,899,987.00",82072 Dawn Centers,Nottingham,England,"99,071.00",UK,19.00,Female,Low,Premium,12/31/2023,"2,023.00",December,8:42:04,2.00,403.35,806.71,Electronics,Samsung,Tablet,Excellent,Standard,Credit Card,Processing,4.00,Lenovo Tab
2,"6,679,610.00","30,192.00",Scott Jensen,Shane85@gmail.com,"8,362,160,449.00",4133 Young Canyon,Geelong,New South Wales,"75,929.00",Australia,48.00,Male,Low,Regular,4/26/2023,"2,023.00",April,4:06:29,3.00,354.48,"1,063.43",Books,Penguin Books,Children's,Average,Same-Day,Credit Card,Processing,2.00,Sports equipment
3,"7,232,460.00","62,101.00",Joseph Miller,Mary34@gmail.com,"2,776,751,724.00",8148 Thomas Creek Suite 100,Edmonton,Ontario,"88,420.00",Canada,56.00,Male,High,Premium,05-08-23,"2,023.00",May,14:55:17,7.00,352.41,"2,466.85",Home Decor,Home Depot,Tools,Excellent,Standard,PayPal,Processing,4.00,Utility knife
4,"4,983,775.00","27,901.00",Debra Coleman,Charles30@gmail.com,"9,098,267,635.00",5813 Lori Ports Suite 269,Bristol,England,"48,704.00",UK,22.00,Male,Low,Premium,01-10-24,"2,024.00",January,16:54:07,2.00,124.28,248.55,Grocery,Nestle,Chocolate,Bad,Standard,Cash,Shipped,1.00,Chocolate cookies


In [ ]:
nb_lignes_depart = retail.shape[0]
nb_lignes_depart

302010

## Sélection des colonnes utiles par date

In [ ]:
# La ville et le pays sont suffisants pour la position géographique
# Les colonnes de mois et année sont parfois fausses, on pourra les créer si besoin
# La catégorie, la marque et le type de produit sont suffisants

retail = retail.drop(['Name','Email','Phone','Address','State','Zipcode','Year','Month','products','Customer_Segment'], axis = 1)

#La colonne Customer_Segment, présente dans le dataset fictif, a été supprimée car sa méthode de calcul était inconnue. Pour garantir la cohérence et la fiabilité de notre analyse,
# nous avons choisi de créer notre propre segmentation RFM, basée sur des règles claires et documentées.

In [ ]:
# Passage de la colonne Date au bon type
# format = 'mixed' pour tenir compte des - et /

retail.Date = pd.to_datetime(retail.Date, format = 'mixed')

## Doublons, NA et types

In [ ]:
# Un exemple pour lequel les valeurs de Transaction_ID et Customer_ID sont identiques
# L'âge et la segmentation sont différentes alors qu'elles ne le devraient pas

retail[retail.Transaction_ID == 6145934.0]

,Transaction_ID,Customer_ID,City,Country,Age,Gender,Income,Date,Time,Total_Purchases,Amount,Total_Amount,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings
140676,"6,145,934.00","76,353.00",Hamilton,Canada,26.00,Male,Medium,2023-11-29,15:48:29,3.00,146.11,438.34,Electronics,Whirepool,Fridge,Good,Express,Credit Card,Delivered,3.00
194626,"6,145,934.00","23,433.00",Bochum,Germany,22.00,Female,Low,2023-03-24,10:38:14,1.00,180.74,180.74,Home Decor,Bed Bath & Beyond,Bedding,Bad,Same-Day,Credit Card,Delivered,1.00
300648,"6,145,934.00","76,353.00",Hamilton,Canada,56.00,Male,Medium,2023-11-29,15:48:29,3.00,146.11,438.34,Clothing,Zara,Dress,Good,Express,Cash,Pending,4.00
301371,"6,145,934.00","23,433.00",Bochum,Germany,22.00,Female,Low,2023-03-24,10:38:14,1.00,180.74,180.74,Home Decor,Bed Bath & Beyond,Bedding,Bad,Same-Day,Cash,Delivered,1.00


In [ ]:
# On vient de voir que les ID de Transaction et de Client ne sont pas uniques
# On supprime les doublons (environ 0,8%) de façon à avoir des combinaisons Transaction ID - Customer ID uniques

retail = retail.drop_duplicates(subset = ['Transaction_ID','Customer_ID'])

print("Le dataframe contient", retail.duplicated().sum(), "doublons.")

Le dataframe contient 0 doublons.


In [ ]:
# On supprime enfin les lignes contenant des valeurs manquantes (environ 1,8%)
# L'index est remis à jour pour pouvoir utiliser les numéros de ligne

retail = retail.dropna(axis = 0, how = 'any').reset_index(drop = True)

In [ ]:
# Conversion en type int

retail.Transaction_ID = retail.Transaction_ID.astype('int')
retail.Customer_ID = retail.Customer_ID.astype('int')
retail.Age = retail.Age.astype('int')
retail.Total_Purchases = retail.Total_Purchases.astype('int')

# Arrondi des colonnes de prix

retail.Amount = round(retail.Amount, 2)
retail.Total_Amount = round(retail.Total_Amount, 2)

**Le client ayant plusieurs attributs différents**

In [ ]:
# Un exemple pour lequel les colonnes City, Country, Age, Gender, Income et Customer_Segment
# sont différentes pour le même Customer_ID

retail[retail.Customer_ID == 37249]

,Transaction_ID,Customer_ID,City,Country,Age,Gender,Income,Date,Time,Total_Purchases,Amount,Total_Amount,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings
0,8691788,37249,Dortmund,Germany,21,Male,Low,2023-09-18,22:03:55,3,108.03,324.09,Clothing,Nike,Shorts,Excellent,Same-Day,Debit Card,Shipped,5.00
9504,2322638,37249,Portsmouth,UK,19,Female,Medium,2023-06-14,21:03:15,7,75.87,531.12,Electronics,Sony,Smartphone,Good,Express,PayPal,Delivered,3.00
60729,1814136,37249,Saskatoon,Canada,20,Male,High,2024-02-06,21:56:51,6,379.38,"2,276.30",Home Decor,Home Depot,Furniture,Average,Express,PayPal,Delivered,2.00
98481,4895457,37249,Hamilton,Canada,46,Male,Low,2023-12-08,14:06:44,4,148.64,594.55,Clothing,Zara,Shirt,Excellent,Express,Cash,Shipped,5.00
201790,5836653,37249,Gold Coast,Australia,29,Male,High,2024-01-02,12:30:45,8,344.92,"2,759.33",Grocery,Pepsi,Water,Average,Same-Day,PayPal,Shipped,2.00


In [ ]:
# On garde la transaction la plus récente, car elle reflète l’état actuel du client. La segmentation RFM et toutes les analyses marketing se basent toujours sur la donnée la plus récente,
#jamais sur la première. Dans un vrai CRM, les Les attributs démographiques (Age, sexe, pays, revenu) sont dans une table client dédiée, donc fixes.

# Trier par Customer_ID et Date (ordre décroissant)
retail = retail.sort_values(["Customer_ID", "Date"], ascending=False)

#Récupérer la transaction la plus récente pour chaque client
cols_fixes = ["Age", "Gender", "Income", "City", "Country"]

last_transaction_info = retail.groupby("Customer_ID").first()[cols_fixes].reset_index()

#Fusionner ces attributs propres dans toutes les lignes
# Retirer les attributs incohérents
retail_clean = retail.drop(columns=cols_fixes)

# Ajouter les attributs corrects basés sur la dernière transaction
retail_clean = retail_clean.merge(last_transaction_info, on="Customer_ID", how="left")

#Vérification de la ligne de la dernère transction
retail_clean[retail_clean["Customer_ID"] == 37249][cols_fixes + ["Customer_ID"]].drop_duplicates()

#Vérification du client 37249
retail_clean[retail_clean["Customer_ID"] == 37249]

,Transaction_ID,Customer_ID,Date,Time,Total_Purchases,Amount,Total_Amount,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings,Age,Gender,Income,City,Country
205061,1814136,37249,2024-02-06,21:56:51,6,379.38,"2,276.30",Home Decor,Home Depot,Furniture,Average,Express,PayPal,Delivered,2.00,20,Male,High,Saskatoon,Canada
205062,5836653,37249,2024-01-02,12:30:45,8,344.92,"2,759.33",Grocery,Pepsi,Water,Average,Same-Day,PayPal,Shipped,2.00,20,Male,High,Saskatoon,Canada
205063,4895457,37249,2023-12-08,14:06:44,4,148.64,594.55,Clothing,Zara,Shirt,Excellent,Express,Cash,Shipped,5.00,20,Male,High,Saskatoon,Canada
205064,8691788,37249,2023-09-18,22:03:55,3,108.03,324.09,Clothing,Nike,Shorts,Excellent,Same-Day,Debit Card,Shipped,5.00,20,Male,High,Saskatoon,Canada
205065,2322638,37249,2023-06-14,21:03:15,7,75.87,531.12,Electronics,Sony,Smartphone,Good,Express,PayPal,Delivered,3.00,20,Male,High,Saskatoon,Canada


In [ ]:
retail_clean.head()

,Transaction_ID,Customer_ID,Date,Time,Total_Purchases,Amount,Total_Amount,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings,Age,Gender,Income,City,Country
0,8452009,99999,2023-09-30,4:40:33,3,365.95,"1,097.84",Books,Random House,Non-Fiction,Average,Standard,Cash,Delivered,2.00,26,Male,High,San Francisco,USA
1,2398254,99999,2023-09-02,13:15:30,9,494.09,"4,446.85",Electronics,Apple,Laptop,Good,Express,PayPal,Processing,4.00,26,Male,High,San Francisco,USA
2,4347464,99998,2024-02-25,17:33:40,3,271.26,813.78,Clothing,Zara,Jeans,Bad,Express,Debit Card,Pending,1.00,59,Female,High,Bonn,Germany
3,1092137,99998,2024-02-03,4:17:07,6,77.31,463.87,Books,Random House,Non-Fiction,Bad,Express,PayPal,Processing,1.00,59,Female,High,Bonn,Germany
4,4700570,99998,2024-01-19,12:45:30,10,314.96,"3,149.65",Home Decor,Home Depot,Tools,Excellent,Standard,PayPal,Processing,5.00,59,Female,High,Bonn,Germany


In [ ]:
# Pour finir, on renomme certaines colonnes pour plus de clarté

retail_clean = retail_clean.rename(columns = {'Total_Purchases' : 'Quantity',
                                  'Amount' : 'Unit Price',
                                  'Total_Amount' : 'Total Price'})

In [ ]:
retail_clean.head()

,Transaction_ID,Customer_ID,Date,Time,Quantity,Unit Price,Total Price,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings,Age,Gender,Income,City,Country
0,8452009,99999,2023-09-30,4:40:33,3,365.95,"1,097.84",Books,Random House,Non-Fiction,Average,Standard,Cash,Delivered,2.00,26,Male,High,San Francisco,USA
1,2398254,99999,2023-09-02,13:15:30,9,494.09,"4,446.85",Electronics,Apple,Laptop,Good,Express,PayPal,Processing,4.00,26,Male,High,San Francisco,USA
2,4347464,99998,2024-02-25,17:33:40,3,271.26,813.78,Clothing,Zara,Jeans,Bad,Express,Debit Card,Pending,1.00,59,Female,High,Bonn,Germany
3,1092137,99998,2024-02-03,4:17:07,6,77.31,463.87,Books,Random House,Non-Fiction,Bad,Express,PayPal,Processing,1.00,59,Female,High,Bonn,Germany
4,4700570,99998,2024-01-19,12:45:30,10,314.96,"3,149.65",Home Decor,Home Depot,Tools,Excellent,Standard,PayPal,Processing,5.00,59,Female,High,Bonn,Germany


**les transactions ayant plusieurs clients**

In [ ]:
# Liste des Transaction_ID ayant plusieurs clients
anomalies = retail_clean.groupby("Transaction_ID")["Customer_ID"].nunique()
anomalies = anomalies[anomalies > 1]
anomalies.head(20)

,Customer_ID
Transaction_ID,
1003092,2
1005669,2
1008244,2
1015985,2
1018230,2
1021291,2
1022174,2
1022229,2
1022661,2


In [ ]:
#afficher toute la ligne
retail_clean[retail_clean["Transaction_ID"].isin(anomalies.index)].sort_values("Transaction_ID").head(10)

,Transaction_ID,Customer_ID,Date,Time,Quantity,Unit Price,Total Price,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings,Age,Gender,Income,City,Country
237488,1003092,27378,2023-05-26,7:06:21,2,262.46,524.91,Grocery,Pepsi,Water,Good,Standard,PayPal,Delivered,4.00,65,Female,High,Manchester,UK
9768,1003092,97124,2023-06-09,1:59:01,6,381.00,"2,285.99",Home Decor,Bed Bath & Beyond,Bathroom,Good,Standard,Cash,Pending,4.00,47,Female,Medium,Stuttgart,Germany
198716,1005669,39211,2024-02-27,2:45:45,7,133.49,934.44,Grocery,Pepsi,Soft Drink,Good,Express,Credit Card,Pending,3.00,22,Female,Medium,Dresden,Germany
207855,1005669,36372,2024-02-10,21:10:21,8,496.80,"3,974.43",Grocery,Nestle,Coffee,Excellent,Same-Day,Debit Card,Delivered,4.00,36,Female,Low,Wollongong,Australia
156258,1008244,52266,2024-01-15,18:10:56,8,383.81,"3,070.50",Books,Penguin Books,Non-Fiction,Bad,Standard,Cash,Shipped,1.00,54,Female,Medium,Sydney,Australia
157412,1008244,51936,2023-03-25,22:33:10,5,107.80,539.02,Grocery,Coca-Cola,Water,Excellent,Standard,Cash,Delivered,4.00,20,Female,Low,Bendigo,Australia
71477,1015985,78164,2023-11-18,1:34:22,3,10.67,32.01,Electronics,Samsung,Tablet,Good,Express,Debit Card,Pending,3.00,23,Male,Medium,New York,USA
279959,1015985,14362,2023-11-16,3:27:07,8,327.21,"2,617.71",Electronics,Sony,Smartphone,Excellent,Same-Day,Cash,Delivered,5.00,22,Male,Medium,Duisburg,Germany
106899,1018230,67337,2023-04-22,3:11:04,9,495.95,"4,463.51",Grocery,Pepsi,Soft Drink,Average,Express,Credit Card,Shipped,2.00,22,Female,Medium,Sheffield,UK
113761,1018230,65244,2024-02-09,16:54:48,10,417.05,"4,170.48",Electronics,Apple,Smartphone,Good,Standard,Debit Card,Delivered,3.00,26,Male,High,Kitchener,Canada


In [ ]:
# Garder seulement la transaction la plus récente pour chaque Transaction_ID
retail_clean = retail_clean.sort_values("Date", ascending=False)
retail_clean = retail_clean.drop_duplicates(subset="Transaction_ID", keep="first")

In [ ]:
# vérification
# aucune transaction n’est associée à plusieurs clients
retail_clean.groupby("Transaction_ID")["Customer_ID"].nunique().max()

1

In [ ]:
nb_lignes_depart = retail.shape[0]
nb_lignes_depart

294399

In [ ]:
nb_lignes_final = retail_clean.shape[0]
nb_lignes_final


289695

**l’analyse RFM**

In [ ]:
#lien de l'article Dinmo qui parle de la RFM : https://www.dinmo.com/fr/segmentation-client/segmentation-rfm/

#Trier la base par date
retail_clean = retail_clean.sort_values(by="Date", ascending=False)
retail_clean.head()

,Transaction_ID,Customer_ID,Date,Time,Quantity,Unit Price,Total Price,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings,Age,Gender,Income,City,Country
91457,5036241,72043,2024-02-29,9:11:33,4,140.59,562.36,Home Decor,Bed Bath & Beyond,Kitchen,Good,Same-Day,Cash,Pending,3.00,26,Male,High,San Francisco,USA
154218,9185908,52894,2024-02-29,1:40:40,1,137.71,137.71,Electronics,Samsung,Television,Average,Express,Cash,Processing,2.00,20,Male,Medium,Chicago,USA
127117,3104718,61157,2024-02-29,6:19:28,2,62.19,124.38,Clothing,Adidas,T-shirt,Good,Express,PayPal,Delivered,3.00,19,Male,Medium,Portsmouth,UK
213355,6209290,34705,2024-02-29,8:00:24,1,369.44,369.44,Grocery,Nestle,Snacks,Good,Express,Credit Card,Delivered,3.00,20,Female,Low,Brighton,UK
233154,3868573,28710,2024-02-29,21:04:03,4,464.67,"1,858.69",Clothing,Nike,Shoes,Average,Same-Day,PayPal,Pending,2.00,64,Female,Low,Launceston,Australia


In [ ]:
#Définir une date de référence (lendemain de la dernière transaction)
ref_date = retail_clean["Date"].max() + pd.Timedelta(days=1)
#Calculer la récence de dernier achat du client, fréquence des transactions et le Montant total dépensé
rfm = (
    retail_clean
    .groupby("Customer_ID")
    .agg({
        "Date": lambda x: (ref_date - x.max()).days,   # Recency
        "Transaction_ID": "nunique",                   # Frequency
        "Total Price": "sum"                           # Monetary
    })
    .reset_index()
)

rfm.columns = ["Customer_ID", "Recency", "Frequency", "Monetary"]


In [ ]:
#Créer les scores R, F, M (1 à 5)

#Score R (récence → inversé)
rfm["R_score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
)

#score F (fréquence → normal)
rfm["F_score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

#Score M (montant → normal)
rfm["M_score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

In [ ]:
#Convertir en entier
rfm[["R_score", "F_score", "M_score"]] = rfm[["R_score", "F_score", "M_score"]].astype(int)

In [ ]:
#Créer MF_score
rfm["FM_product"] = rfm["F_score"] * rfm["M_score"]

rfm["FM_score"] = pd.qcut(
    rfm["FM_product"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

#convertir en entier
rfm["FM_score"] = rfm["FM_score"].astype(int)

In [ ]:
#Attribuer un segment RFM (matrice)
seg_matrice = {
    (5,1): "Cannot Lose Them",
    (5,2): "Cannot Lose Them",
    (5,3): "Loyal",
    (5,4): "Champions",
    (5,5): "Champions",

    (4,1): "At Risk",
    (4,2): "At Risk",
    (4,3): "Loyal",
    (4,4): "Champions",
    (4,5): "Champions",

    (3,1): "At Risk",
    (3,2): "At Risk",
    (3,3): "Need Attention",
    (3,4): "Potential Loyalists",
    (3,5): "Potential Loyalists",

    (2,1): "Lost",
    (2,2): "Lost",
    (2,3): "About to Sleep",
    (2,4): "Potential Loyalists",
    (2,5): "Potential Loyalists",

    (1,1): "Lost",
    (1,2): "Lost",
    (1,3): "About to Sleep",
    (1,4): "Promising",
    (1,5): "New",
}

#Appliquer le segment
rfm["Segment"] = rfm.apply(lambda row: seg_matrice.get((row["FM_score"], row["R_score"]), "Other"), axis=1)

In [ ]:
rfm.head()

,Customer_ID,Recency,Frequency,Monetary,R_score,F_score,M_score,FM_product,FM_score,Segment
0,10000,103,4,"5,007.57",2,3,4,12,3,At Risk
1,10001,105,4,"7,167.13",2,3,5,15,4,At Risk
2,10002,95,5,"4,104.02",3,4,3,12,3,Need Attention
3,10003,228,2,"2,340.50",1,1,2,2,1,Lost
4,10004,31,2,"2,356.52",4,1,2,2,1,Promising


In [ ]:
#Tableau de synthèse final
segment_summary = (
    rfm.groupby("Segment")
    .agg(
        nb_clients=("Customer_ID", "nunique"),
        nb_transactions=("Frequency", "sum"),
        revenue_total=("Monetary", "sum"),
        avg_recency=("Recency", "mean"),
        avg_frequency=("Frequency", "mean"),
        avg_revenue=("Monetary", "mean")
    )
    .reset_index()
)

segment_summary["revenue_total"] = segment_summary["revenue_total"].astype(float).round(2)
segment_summary["avg_recency"] = segment_summary["avg_recency"].astype(float).round().astype(int)
segment_summary["avg_frequency"] = segment_summary["avg_frequency"].astype(float).round().astype(int)
segment_summary["avg_revenue"] = segment_summary["avg_revenue"].astype(float).round(2)

segment_summary = segment_summary.sort_values("revenue_total", ascending=False)

segment_summary



,Segment,nb_clients,nb_transactions,revenue_total,avg_recency,avg_frequency,avg_revenue
3,Champions,18573,94950,"140,925,744.09",24,5,"7,587.67"
1,At Risk,11583,39164,"58,004,522.30",161,3,"5,007.73"
5,Loyal,7714,37105,"56,770,826.78",74,5,"7,359.45"
8,Potential Loyalists,12413,37442,"42,657,333.99",26,3,"3,436.50"
4,Lost,19772,34010,"37,780,471.93",202,2,"1,910.81"
2,Cannot Lose Them,3199,16823,"27,879,692.59",142,5,"8,715.13"
6,Need Attention,3736,12222,"15,251,318.16",74,3,"4,082.26"
0,About to Sleep,5827,12155,"12,267,142.87",75,2,"2,105.22"
9,Promising,1961,3178,"2,598,221.08",39,2,"1,324.95"
7,New,1607,2646,"2,168,514.29",12,2,"1,349.42"


In [ ]:
# Sauvegarde au format CSV
retail_clean.to_csv(chemin + "retail_clean.csv", index=False)
rfm.to_csv(chemin + "rfm_client_table.csv", index=False)